# NeoNatal Watch AI — Phase 9: Model Fusion (Ensemble)

> **DEMO / SYNTHETIC DATA — NOT FOR CLINICAL USE**
>
> In this phase, we combine all 4 of our models (XGBoost, CNN-LSTM, Autoencoder, Transformer) into a single, unified risk score.

---

## Why Fuse Models?
- **XGBoost**: Excellent precision, but sometimes misses early warning signs (low recall).
- **CNN-LSTM**: Good balance of precision and recall; great at reading sequential trends.
- **Autoencoder**: Extremely sensitive; acts as a 'tripwire' for any anomaly, but produces false alarms.
- **Transformer**: Adds parallel self-attention context.

By taking a weighted average of their predictions, we smooth out individual weaknesses and boost overall clinical reliability.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ml.models.fusion import FusionModel
from ml.evaluation.metrics import evaluate_model

print('Imports OK')

In [ ]:
# Load Test Data (Both tabular and 3D sequences)
test_flat = pd.read_csv('../data/processed/test_features.csv')
X_test_seq = np.load('../data/processed/X_test.npy')
y_test = np.load('../data/processed/y_test.npy')

# Align the flat features to exactly match the sequence windows
aligned_flat = []
for pid, grp in test_flat.groupby('patient_id'):
    aligned_flat.append(grp.iloc[29:-1])
test_flat = pd.concat(aligned_flat, ignore_index=True)

print(f"Flat features shape: {test_flat.shape}")
print(f"Sequence shape: {X_test_seq.shape}")

In [ ]:
# Initialize the Fusion Model
# It automatically loads XGBoost, CNN-LSTM, Transformer, and Autoencoder
fusion = FusionModel(models_dir='../models')
print(f"Fusion Weights: {fusion.weights}")

In [ ]:
# Generate Unified Predictions
fusion_probs, individual_probs = fusion.predict_risk(
    flat_features=test_flat, 
    sequence_features=X_test_seq
)

In [ ]:
# Evaluate the Ensemble Model
metrics = evaluate_model(
    y_true=y_test,
    y_probs=fusion_probs,
    model_name="Fusion_Ensemble_Notebook",
    save_dir="../reports/figures"
)

In [ ]:
# Plot a comparison of PR-AUC across all models
from sklearn.metrics import average_precision_score
comparison = {name: average_precision_score(y_test, probs) for name, probs in individual_probs.items()}
comparison['FUSION'] = metrics['pr_auc']

plt.figure(figsize=(10, 6))
colors = ['steelblue'] * 4 + ['crimson']
plt.bar(comparison.keys(), comparison.values(), color=colors)
plt.title('PR-AUC Model Comparison')
plt.ylabel('PR-AUC Score')
plt.show()